# Re-EDA — identidad, nombres, marcas y presentaciones

**Snapshot analizado:** `off_csv_20260919` (mismo universo que `notebooks/02_eda_universo_mexico.ipynb`: export CSV de Open Food Facts filtrado a México).

**Universo:** 16.851 productos en `datos/procesados/off_mexico_20260919.parquet`. Este notebook **no modifica** ese Parquet ni `matriz_nut_100g_20260919.parquet`.

**Fecha de ejecución:** 2026-09-26.

**Objetivo:** cubrir los huecos que el EDA del 2026-09-20 no midió de forma sistemática — EAN/GTIN, nombres residuales, marcas, `quantity` y duplicados de texto — y dejar tablas CSV citables. No se reescribe el notebook 02 (nutrientes, NOVA, alérgenos, categoría). No se llama a APIs externas (A37 paso 4 es un experimento aparte). No se cambia el ranking.

**Decisiones ya cerradas que este notebook respeta:** A28 (fallback de nombre interno), A33 (procedencia), A34 (llave = GTIN, no agrupar presentaciones), A37 (no imputar nombres).

**Regla de trabajo:** un campo que existe no se asume completo. Toda cobertura se reporta sobre el universo total (`N` medido en la primera celda), nunca sobre el subconjunto que ya tenía el dato.

**Aviso:** es diagnóstico de datos. No contiene evaluación médica ni recomendaciones de compra.


In [ ]:
"""Configuración: DuckDB sobre el Parquet México, sin cargarlo entero salvo tablas chicas."""

import json
import re
from pathlib import Path

import duckdb
import pandas as pd

import nutrimatch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SNAPSHOT_ID = "off_csv_20260919"
PARQUET_PATH = REPO_ROOT / "datos" / "procesados" / "off_mexico_20260919.parquet"
META_PATH = REPO_ROOT / "datos" / "snapshots" / SNAPSHOT_ID / "_metadata.json"
OUT_DIR = REPO_ROOT / "datos" / "procesados" / "re_eda_20260926"

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 140)

con = duckdb.connect()
P = PARQUET_PATH.as_posix()
metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
N = con.execute(f"SELECT count(*) FROM '{P}'").fetchone()[0]


def cobertura(campo: str) -> int:
    """Filas con texto o número no vacío (trim). No trata 'unknown' como ausencia."""
    return con.execute(
        f"""SELECT count(*) FROM '{P}'
            WHERE "{campo}" IS NOT NULL
              AND trim(cast("{campo}" AS VARCHAR)) NOT IN ('', 'nan', 'NaN', 'None')"""
    ).fetchone()[0]


print(f"nutrimatch versión: {nutrimatch.__version__}")
print(f"snapshot_id: {metadata['snapshot_id']}")
print(f"generado_en: {metadata['generado_en']}")
print(f"fuente export: {metadata['procedencia']['url']}")
print(f"sha256 export: {metadata['procedencia']['sha256'][:16]}…")
print(f"Universo cargado: N = {N:,} productos")
print(f"Parquet (solo lectura): {PARQUET_PATH}")


### Hallazgo 0 — el universo no se movió

`N = 16.851`, idéntico a la ingesta y al notebook 02. `snapshot_id = off_csv_20260919`. Si esta celda diera otro conteo, el resto del notebook no sería comparable al diagnóstico del 2026-09-26 y habría que parar.


## 1. EAN / GTIN

`code` es la llave (A34). Aquí se mide si sirve como GTIN de cruce externo: solo dígitos, longitud GS1 (8/12/13/14) y checksum. La bandera `gtin_valid` se calcula en memoria; **no** se escribe en el snapshot.


In [ ]:
def gtin_checksum_ok(code: str) -> bool:
    """Checksum GS1 sobre GTIN-8/12/13/14 (se rellena a 14 dígitos por la izquierda)."""
    s = str(code).strip()
    if not s.isdigit() or len(s) not in (8, 12, 13, 14):
        return False
    padded = s.zfill(14)
    digits = [int(c) for c in padded]
    check, body = digits[-1], digits[:-1]
    total = 0
    for i, d in enumerate(reversed(body)):
        total += d * (3 if i % 2 == 0 else 1)
    return ((10 - (total % 10)) % 10) == check


df_codes = con.execute(f"SELECT trim(code) AS code FROM '{P}'").df()
df_codes["len"] = df_codes["code"].str.len()
df_codes["gtin_valid"] = df_codes["code"].map(gtin_checksum_ok)

gtin_longitud = (
    df_codes["len"]
    .value_counts()
    .sort_index()
    .rename_axis("longitud")
    .reset_index(name="n")
)
gtin_longitud["pct"] = (gtin_longitud["n"] * 100 / N).round(2)
gtin_longitud["longitud_gs1"] = gtin_longitud["longitud"].isin([8, 12, 13, 14])

n_no_digito = int((~df_codes["code"].str.fullmatch(r"\d+")).sum())
n_gtin_valid = int(df_codes["gtin_valid"].sum())
prefijo_13 = (
    df_codes.loc[df_codes["len"] == 13, "code"]
    .str[:3]
    .value_counts()
    .head(10)
    .rename_axis("prefijo")
    .reset_index(name="n")
)

print("no numéricos:", n_no_digito)
print(f"gtin_valid: {n_gtin_valid:,} ({n_gtin_valid * 100 / N:.2f} %)")
print(f"gtin inválido o longitud no GS1: {N - n_gtin_valid:,}")
print("\nLongitud de code:")
print(gtin_longitud.to_string(index=False))
print("\nPrefijos de los GTIN-13 (top 10):")
print(prefijo_13.to_string(index=False))
print(f"\nPrefijo 750 (GS1 México) entre GTIN-13: "
      f"{int((df_codes.loc[df_codes['len']==13, 'code'].str[:3]=='750').sum()):,}")


### Hallazgo 1 — `code` es usable como GTIN, con una cola sucia del 3,1 %

Medido el 2026-09-26 sobre N = 16.851:

- 0 códigos no numéricos.
- 16.414 (97,41 %) tienen longitud 13; 415 (2,46 %) longitud 8; 11 longitud 14.
- 11 códigos tienen longitud rara (15–38): no son GTIN.
- **Checksum GS1 válido: 16.331 (96,91 %).** Los 520 restantes no deberían usarse como llave de cruce externo sin revisión.
- Entre los GTIN-13, **11.247 (68,52 % de los de 13 dígitos)** empiezan con `750` (prefijo GS1 México). El resto son sobre todo prefijos de EE.UU. (`004`, `007`, `001`…) e importados (`841` España).

`gtin_valid` queda como candidata para el dataset analítico (paso H.9 del diagnóstico). No se materializa en el Parquet crudo.


## 2. Nombres

Tres campos del **mismo** registro OFF: `product_name`, `generic_name`, `abbreviated_product_name`. A28 ya define el fallback interno. Aquí se mide cuánto recupera y qué queda en NULL genuino (A37).


In [ ]:
nombres_combinaciones = con.execute(f"""
WITH t AS (
  SELECT
    (product_name IS NOT NULL AND trim(product_name) <> '') AS has_product_name,
    (generic_name IS NOT NULL AND trim(generic_name) <> '') AS has_generic_name,
    (abbreviated_product_name IS NOT NULL AND trim(abbreviated_product_name) <> '') AS has_abbreviated
  FROM '{P}'
)
SELECT has_product_name, has_generic_name, has_abbreviated,
       count(*) AS n,
       round(100.0 * count(*) / {N}, 2) AS pct
FROM t
GROUP BY 1, 2, 3
ORDER BY n DESC
""").df()

a28 = con.execute(f"""
SELECT
  sum(CASE WHEN product_name IS NOT NULL AND trim(product_name) <> '' THEN 1 ELSE 0 END) AS desde_product_name,
  sum(CASE WHEN (product_name IS NULL OR trim(product_name) = '')
            AND generic_name IS NOT NULL AND trim(generic_name) <> '' THEN 1 ELSE 0 END) AS fallback_generic_name,
  sum(CASE WHEN (product_name IS NULL OR trim(product_name) = '')
            AND (generic_name IS NULL OR trim(generic_name) = '')
            AND abbreviated_product_name IS NOT NULL AND trim(abbreviated_product_name) <> '' THEN 1 ELSE 0 END) AS fallback_abbreviated,
  sum(CASE WHEN (product_name IS NULL OR trim(product_name) = '')
            AND (generic_name IS NULL OR trim(generic_name) = '')
            AND (abbreviated_product_name IS NULL OR trim(abbreviated_product_name) = '') THEN 1 ELSE 0 END) AS sin_ningun_nombre
FROM '{P}'
""").df()

sin_nombre_cobertura = con.execute(f"""
SELECT
  count(*) AS n_sin_nombre,
  sum(CASE WHEN brands IS NOT NULL AND trim(brands) <> '' THEN 1 ELSE 0 END) AS con_marca,
  sum(CASE WHEN image_url IS NOT NULL AND trim(image_url) <> '' THEN 1 ELSE 0 END) AS con_imagen,
  sum(CASE WHEN ingredients_text IS NOT NULL AND trim(ingredients_text) <> '' THEN 1 ELSE 0 END) AS con_ingredientes,
  sum(CASE WHEN quantity IS NOT NULL AND trim(quantity) <> '' THEN 1 ELSE 0 END) AS con_quantity
FROM '{P}'
WHERE (product_name IS NULL OR trim(product_name) = '')
  AND (generic_name IS NULL OR trim(generic_name) = '')
  AND (abbreviated_product_name IS NULL OR trim(abbreviated_product_name) = '')
""").df()

nombres = con.execute(f"""
SELECT product_name
FROM '{P}'
WHERE product_name IS NOT NULL AND trim(product_name) <> ''
""").df()["product_name"]


def es_mayusculas(texto: str) -> bool:
    letras = [c for c in str(texto) if c.isalpha()]
    return len(letras) >= 4 and all(c.isupper() for c in letras)


n_mayus = int(nombres.map(es_mayusculas).sum())
n_no_ascii = int(nombres.map(lambda s: any(ord(c) > 127 for c in str(s))).sum())

print("Combinaciones de los tres campos:")
print(nombres_combinaciones.to_string(index=False))
print("\nResolución A28 (orden product_name → generic_name → abbreviated):")
print(a28.to_string(index=False))
print("\nDe los que no tienen ningún nombre, qué otra cosa sí tienen:")
print(sin_nombre_cobertura.to_string(index=False))
print(f"\nSobre {len(nombres):,} product_name no vacíos:")
print(f"  MAYÚSCULAS (solo letras, ≥4): {n_mayus:,} ({n_mayus * 100 / len(nombres):.1f} %)")
print(f"  algún caracter no ASCII:     {n_no_ascii:,} ({n_no_ascii * 100 / len(nombres):.1f} %)")

placeholder = con.execute(f"""
SELECT lower(trim(product_name)) AS nombre, count(*) AS n
FROM '{P}'
WHERE lower(trim(product_name)) IN ('cargando…', 'cargando...', 'cargando')
GROUP BY 1
""").df()
print("\nPlaceholders de captura en product_name:")
print(placeholder.to_string(index=False) if len(placeholder) else "(ninguno)")


### Hallazgo 2 — A28 recupera 68 nombres; 1.672 (9,92 %) no tienen ninguno

Combinaciones (N = 16.851):

| product_name | generic_name | abbreviated | n | % |
| --- | --- | --- | --- | --- |
| sí | no | no | 10.909 | 64,74 |
| sí | sí | no | 4.157 | 24,67 |
| **no** | **no** | **no** | **1.672** | **9,92** |
| no | sí | sí | 66 | 0,39 |
| sí | sí | sí | 44 | 0,26 |
| resto |  |  | 3 | 0,02 |

A28 aplicado:

- 15.111 desde `product_name`;
- 68 fallback a `generic_name`;
- 0 fallback solo a `abbreviated_product_name` (los 66 abreviados ya tenían `generic_name`);
- **1.672 sin ningún nombre** — NULL genuino, no se inventa (A2, A37).

De esos 1.672: **1.496 tienen `image_url`**, 557 marca, 261 `quantity`, 209 ingredientes. El producto existe; el contribuidor no escribió el título. Es el conjunto candidato al experimento de API OFF por EAN (A37 paso 4), no a un modelo que invente strings.

Sobre los 15.111 con `product_name`: 709 (4,7 %) están en MAYÚSCULAS; 2.344 (15,5 %) tienen caracteres no ASCII. La normalización cosmética (A37 paso 3) es DERIVED y no pisa el original.

Aparece además un error de captura: **14 filas tienen `product_name = "cargando…"`**. No es un nombre de producto; es un placeholder de la UI de contribución que se grabó en el export. Candidato a tratar como NULL en la homologación (H.3), con el original conservado.


## 3. Duplicados de texto (no de `code`)

A34: no se colapsan presentaciones. Aquí solo se cuenta cuántos textos se repiten y se muestra una muestra de colisiones marca+nombre.


In [ ]:
dup_nombre = con.execute(f"""
WITH t AS (
  SELECT lower(trim(product_name)) AS nombre
  FROM '{P}'
  WHERE product_name IS NOT NULL AND trim(product_name) <> ''
)
SELECT count(*) AS nombres_no_nulos,
       count(DISTINCT nombre) AS unicos,
       count(*) - count(DISTINCT nombre) AS repeticiones
FROM t
""").df()

top_nombres = con.execute(f"""
SELECT lower(trim(product_name)) AS nombre, count(*) AS n
FROM '{P}'
WHERE product_name IS NOT NULL AND trim(product_name) <> ''
GROUP BY 1
HAVING count(*) >= 10
ORDER BY n DESC
LIMIT 15
""").df()

dup_marca_nombre = con.execute(f"""
WITH t AS (
  SELECT lower(trim(coalesce(brands, ''))) AS marca,
         lower(trim(coalesce(product_name, ''))) AS nombre
  FROM '{P}'
  WHERE product_name IS NOT NULL AND trim(product_name) <> ''
)
SELECT count(*) AS pares,
       count(DISTINCT marca || '|' || nombre) AS unicos,
       count(*) - count(DISTINCT marca || '|' || nombre) AS repeticiones
FROM t
""").df()

colisiones = con.execute(f"""
SELECT lower(trim(brands)) AS marca,
       lower(trim(product_name)) AS nombre,
       count(*) AS n,
       string_agg(code, ', ') AS codes
FROM '{P}'
WHERE product_name IS NOT NULL AND trim(product_name) <> ''
  AND brands IS NOT NULL AND trim(brands) <> ''
GROUP BY 1, 2
HAVING count(*) >= 3
ORDER BY n DESC
LIMIT 12
""").df()

print("Nombres (lower + trim):")
print(dup_nombre.to_string(index=False))
print("\nNombres repetidos ≥10 veces:")
print(top_nombres.to_string(index=False))
print("\nPares marca + nombre:")
print(dup_marca_nombre.to_string(index=False))
print("\nColisiones marca+nombre con ≥3 codes distintos (muestra):")
print(colisiones.to_string(index=False))


### Hallazgo 3 — los textos se repiten; los `code` no. No se agrupa.

- 15.111 nombres no nulos → 12.444 únicos → **2.667 repeticiones**.
- Top: “queso panela” (22), “tostadas” (20), “queso manchego” (19). Son categorías genéricas, no el mismo GTIN.
- Pares marca+nombre: 15.111 → 14.226 únicos → **885 repeticiones**.
- Muestra de colisiones (misma marca y mismo nombre, distinto `code`): Knorr “caldo de pollo” (9 codes), Nestlé/Nesquik (6), Kellogg's Zucaritas (6), Sabritas Rancheritos (6). Son presentaciones (tamaño, receta, mercado) o GTIN de caja vs unidad.

Confirma A34: colapsar por texto mezclaría observaciones distintas. El objeto de análisis sigue siendo el GTIN.


## 4. Marcas


In [ ]:
n_brands = cobertura("brands")
n_brands_tags = cobertura("brands_tags")
n_owner = cobertura("brand_owner")

marcas_top = con.execute(f"""
SELECT lower(trim(brands)) AS marca, count(*) AS n
FROM '{P}'
WHERE brands IS NOT NULL AND trim(brands) <> ''
GROUP BY 1
ORDER BY n DESC
LIMIT 15
""").df()
marcas_top["pct_universo"] = (marcas_top["n"] * 100 / N).round(2)

nestle = con.execute(f"""
SELECT lower(trim(brands)) AS marca, count(*) AS n
FROM '{P}'
WHERE lower(brands) LIKE '%nestl%'
GROUP BY 1
ORDER BY n DESC
LIMIT 12
""").df()

print(f"brands:      {n_brands:,} ({n_brands * 100 / N:.1f} %)")
print(f"brands_tags: {n_brands_tags:,} ({n_brands_tags * 100 / N:.1f} %)  — misma cobertura que brands")
print(f"brand_owner: {n_owner:,} ({n_owner * 100 / N:.1f} %)")
print("\nTop 15 marcas (texto lower/trim, sin homologar):")
print(marcas_top.to_string(index=False))
print("\nVariantes que contienen 'nestl':")
print(nestle.to_string(index=False))


### Hallazgo 4 — 22,2 % sin marca; `nestlé` y `nestle` son dos llaves

- `brands` y `brands_tags` cubren **13.102 (77,8 %)** — la misma N: no hay marca en uno y no en el otro.
- `brand_owner` solo 370 (2,2 %): no es fuente de homologación.
- Top: Great Value 361 (marca propia Walmart), Lala 220, Bimbo 173, Nestlé 140, Danone 138.
- **`nestlé` = 140 y `nestle` = 106** son la misma empresa escrita distinto. Cualquier cruce QQP por texto (A35) tiene que homologar marcas *antes* del match, o el join fallará en silencio.

Homologar marcas es el paso H.3 del diagnóstico (DERIVED, original intacto). Este notebook no lo aplica.


## 5. Presentación: `quantity` y `product_quantity`


In [ ]:
n_qty = cobertura("quantity")
n_pq = cobertura("product_quantity")

qty_cruce = con.execute(f"""
SELECT
  sum(CASE WHEN quantity IS NOT NULL AND trim(quantity) <> ''
            AND product_quantity IS NOT NULL AND trim(cast(product_quantity AS VARCHAR)) <> '' THEN 1 ELSE 0 END) AS ambos,
  sum(CASE WHEN quantity IS NOT NULL AND trim(quantity) <> ''
            AND (product_quantity IS NULL OR trim(cast(product_quantity AS VARCHAR)) = '') THEN 1 ELSE 0 END) AS solo_quantity,
  sum(CASE WHEN (quantity IS NULL OR trim(quantity) = '')
            AND product_quantity IS NOT NULL AND trim(cast(product_quantity AS VARCHAR)) <> '' THEN 1 ELSE 0 END) AS solo_product_quantity
FROM '{P}'
""").df()

qdf = con.execute(f"""
SELECT quantity
FROM '{P}'
WHERE quantity IS NOT NULL AND trim(quantity) <> ''
""").df()["quantity"].astype(str)

UNIDADES = ("kg", "ml", "fl oz", "oz", "lb", "piezas", "unidades", "pza", "pack", "g", "l")
conteo_unidades = {u: 0 for u in UNIDADES}
sin_espacio = 0
con_digito = 0
for bruto in qdf:
    sl = bruto.lower().strip()
    if re.search(r"\d", sl):
        con_digito += 1
    if re.search(r"\d+(g|ml|kg|l)\b", sl) and not re.search(r"\d+\s+(g|ml|kg|l)\b", sl):
        sin_espacio += 1
    for u in UNIDADES:
        if u == "l":
            if re.search(r"(?<![a-z])l(?![a-z])", sl) and "ml" not in sl:
                conteo_unidades[u] += 1
        elif u == "g":
            if re.search(r"(?<![a-z])g(?![a-z])", sl) and "kg" not in sl:
                conteo_unidades[u] += 1
        elif re.search(rf"(?<![a-z]){re.escape(u)}(?![a-z])", sl):
            conteo_unidades[u] += 1

quantity_unidades = pd.DataFrame(
    {"unidad": list(conteo_unidades), "n": list(conteo_unidades.values())}
).sort_values("n", ascending=False)
quantity_unidades["pct_con_quantity"] = (quantity_unidades["n"] * 100 / n_qty).round(1)

ejemplos = con.execute(f"""
SELECT quantity, count(*) AS n
FROM '{P}'
WHERE quantity IS NOT NULL AND trim(quantity) <> ''
GROUP BY 1
ORDER BY n DESC
LIMIT 12
""").df()

print(f"quantity:          {n_qty:,} ({n_qty * 100 / N:.1f} %)")
print(f"product_quantity:  {n_pq:,} ({n_pq * 100 / N:.1f} %)")
print(qty_cruce.to_string(index=False))
print(f"\nDe {n_qty:,} quantity: {con_digito:,} traen al menos un dígito; "
      f"{sin_espacio:,} pegan número y unidad (`200g`).")
print("\nUnidades detectadas (pueden solaparse en un mismo texto):")
print(quantity_unidades.to_string(index=False))
print("\nValores más frecuentes de quantity:")
print(ejemplos.to_string(index=False))


### Hallazgo 5 — la presentación es texto libre; `product_quantity` no añade filas

- `quantity`: **8.001 (47,5 %)**. `product_quantity`: **7.706 (45,7 %)**.
- Cruce: 7.706 tienen ambos; **295 solo `quantity`**; **0 solo `product_quantity`**. El campo numérico de OFF es un subconjunto del texto, no una fuente extra.
- De los 8.001 con texto, 7.985 traen un dígito. **1.714 pegan número y unidad** (`200g` vs `200 g`): hay que parsear las dos formas.
- Unidades (aproximadas, con frontera de palabra): g 5.296, ml 1.120, L 528, kg 302, oz 193, piezas 43, lb 29. Mezcla métrico e imperial.
- Valores típicos: `500 g`, `400 g`, `1 l` / `1 L` (la L mayúscula y minúscula no están unificadas).

Homologar a número + unidad canónica es previo a cualquier precio por 100 g. No se aplica en este corte. El faltante (52,5 %) queda UNAVAILABLE, no se imputa (A36).


## 6. Anexo — `stores` y sellos en `labels_tags`

No son identidad. El diagnóstico (H.1) los pidió y el notebook 02 no los desglosó así. Cobertura sobre N, no sobre quienes ya tenían el campo.


In [ ]:
n_stores = cobertura("stores")
n_labels = cobertura("labels_tags")

sellos = con.execute(f"""
SELECT
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'es:exceso') THEN 1 ELSE 0 END) AS exceso_nom051,
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'en:organic') THEN 1 ELSE 0 END) AS organic,
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'en:no-gluten')
            OR contains(lower(coalesce(labels_tags, '')), 'en:gluten-free') THEN 1 ELSE 0 END) AS no_gluten,
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'en:fair-trade') THEN 1 ELSE 0 END) AS fair_trade,
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'en:vegetarian') THEN 1 ELSE 0 END) AS vegetarian_sello,
  sum(CASE WHEN contains(lower(coalesce(labels_tags, '')), 'en:vegan') THEN 1 ELSE 0 END) AS vegan_sello
FROM '{P}'
""").df()

stores_top = con.execute(f"""
SELECT lower(trim(stores)) AS stores, count(*) AS n
FROM '{P}'
WHERE stores IS NOT NULL AND trim(stores) <> ''
GROUP BY 1
ORDER BY n DESC
LIMIT 10
""").df()

print(f"stores:      {n_stores:,} ({n_stores * 100 / N:.1f} %)")
print(f"labels_tags: {n_labels:,} ({n_labels * 100 / N:.1f} %)")
print("\nSellos (un producto puede contar en más de una columna):")
print(sellos.to_string(index=False))
print("\nstores más frecuentes (texto crudo, a menudo listas):")
print(stores_top.to_string(index=False))


### Hallazgo 6 — `stores` no es un precio; los sellos cubren poco

- `stores`: 4.564 (27,1 %). El valor más frecuente es el literal `walmart` (1.118). El resto son listas (`walmart,sams club`). Es mención de cadena, **no** un precio (A5, A35). No se usa para imputar ni para puntuar.
- `labels_tags`: 4.266 (25,3 %), igual que en el notebook 02. Dentro de ese cuarto: `es:exceso-*` 1.201 (7,1 % del universo), sin gluten 676, vegetariano 325, vegano 304, orgánico 170, comercio justo 24.
- D3 solo puede acertar un sello si el producto lo trae en `labels_tags`. Si la usuaria valora `en:organic`, el máximo de éxitos posibles es 170 productos, no 16.851.


## 7. Exportar tablas resumen

CSV chicos, citables. No se escribe ninguna columna nueva en el snapshot. Re-ejecutar esta celda regenera `datos/procesados/re_eda_20260926/`.


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

gtin_longitud.to_csv(OUT_DIR / "gtin_longitud.csv", index=False)
nombres_combinaciones.to_csv(OUT_DIR / "nombres_combinaciones.csv", index=False)
sin_nombre_cobertura.to_csv(OUT_DIR / "sin_nombre_cobertura.csv", index=False)
marcas_top.to_csv(OUT_DIR / "marcas_top.csv", index=False)
quantity_unidades.to_csv(OUT_DIR / "quantity_unidades.csv", index=False)

print("Escrito en", OUT_DIR)
for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {p.name:30s} {p.stat().st_size} bytes")


## 8. Cierre — sin decisión nueva

Este re-EDA **confirma** las cifras del diagnóstico del 2026-09-26 (1.672 sin nombre, checksum 96,91 %, `quantity` 47,5 %, 885 pares marca+nombre repetidos). No se reabre A33–A37.

Hallazgos adicionales que no cambian una decisión, pero sí el trabajo de homologación (H.3):

1. 14 `product_name = "cargando…"` son placeholders, no nombres.
2. `nestlé` / `nestle` hay que unificar antes de cualquier cruce por texto.
3. `product_quantity` no cubre filas que `quantity` no cubra.
4. `gtin_valid` es una bandera candidata para el dataset analítico, no para el Parquet crudo.

Siguiente paso, cuando se pida: homologar nombres y marcas en un Parquet **nuevo** (H.3), o el experimento de API OFF sobre una muestra de los 1.672 (A37), no la interfaz.
